# DSS Benchmark: SFARI EEG Multi-Paradigm Dataset

This notebook benchmarks **mne-denoise DSS** methods against the original preprocessing
and analysis pipelines from two studies using the SFARI EEG multi-paradigm dataset
(OpenNeuro ds006780):

- **FAST**: Face processing oddball task (Vanneau, Brittenham et al. 2025)
- **AVSRT**: Audiovisual Simple Reaction Time (Vanneau, Foxe et al. 2025)

## Outline

| Section | Content |
|---------|--------|
| 0 | Setup & dependencies |
| 1 | Download dataset via DataLad |
| 2 | Reproduce original preprocessing |
| 3 | Reproduce FAST analysis (ERPs, theta, alpha, gamma ITPC) |
| 4 | Reproduce AVSRT analysis (ERPs, alpha MSI, wPLI) |
| 5 | DSS Usage 1: Artifact removal (replaces ICA) |
| 6 | DSS Usage 2: Line noise removal |
| 7 | DSS Usage 3: ERP enhancement |
| 8 | DSS Usage 4: Alpha band extraction |
| 9 | DSS Usage 5: Theta band extraction |
| 10 | DSS Usage 6: Gamma ITPC enhancement |
| 11 | DSS Usage 7: Connectivity enhancement (wPLI) |
| 12 | Benchmark summary |

---
## Section 0: Setup & Dependencies

In [42]:
import os
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import mne_bids
import numpy as np
import pandas as pd
from scipy import signal, stats

# mne-denoise imports
from mne_denoise.dss import (
    DSS,
    AverageBias,
    BandpassBias,
    CycleAverageBias,
    GaussDenoiser,
    IterativeDSS,
    KurtosisDenoiser,
    LineNoiseBias,
    RobustTanhDenoiser,
    SmoothingBias,
    SpectrogramDenoiser,
    TanhMaskDenoiser,
    WienerMaskDenoiser,
    beta_gauss,
    beta_pow3,
    beta_tanh,
    narrowband_dss,
    narrowband_scan,
    time_shift_dss,
)
from mne_denoise.zapline import ZapLine

# Preprocessing tools matching original pipeline
try:
    from pyprep.find_noisy_channels import NoisyChannels
    HAS_PYPREP = True
except ImportError:
    HAS_PYPREP = False
    warnings.warn("pyprep not installed. Bad channel detection will be skipped.")

try:
    from mne_icalabel import label_components
    HAS_ICALABEL = True
except ImportError:
    HAS_ICALABEL = False
    warnings.warn("mne-icalabel not installed. ICLabel classification unavailable.")

try:
    from mne_connectivity import spectral_connectivity_epochs
    HAS_CONNECTIVITY = True
except ImportError:
    HAS_CONNECTIVITY = False
    warnings.warn("mne-connectivity not installed. wPLI analysis unavailable.")

plt.style.use("seaborn-v0_8-whitegrid")
mne.set_log_level("WARNING")

print(f"MNE version: {mne.__version__}")
print(f"pyprep available: {HAS_PYPREP}")
print(f"mne-icalabel available: {HAS_ICALABEL}")
print(f"mne-connectivity available: {HAS_CONNECTIVITY}")

MNE version: 1.9.0
pyprep available: True
mne-icalabel available: True
mne-connectivity available: True


In [27]:
# ============================================================
# All parameters inline - no config files
# ============================================================

# Paths
DATASET_DIR = Path("data/ds006780")  # DataLad clone target
RESULTS_DIR = Path("results/sfari_benchmark")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Dataset parameters
SFREQ = 512  # BioSemi sampling rate
N_CHANNELS = 64  # BioSemi 64-channel cap
LINE_FREQ = 60  # US power line frequency (Albert Einstein College, NYC)

# Subject selection: 5 per group for development
# These will be populated after reading participants.tsv
N_SUBJECTS_PER_GROUP = 1

# Preprocessing parameters (exact match to papers)
BAD_CHANNEL_THRESHOLD = 0.15  # Reject participant if >15% bad channels
FILTER_LOW = 0.01  # Hz, FIR bandpass low cutoff
FILTER_HIGH = 40.0  # Hz, FIR bandpass high cutoff
ICA_HIGHPASS = 1.0  # Hz, highpass for ICA fitting

# FAST paradigm parameters
FAST_TMIN = -0.5  # Epoch start (s)
FAST_TMAX = 1.0  # Epoch end (s)
FAST_BASELINE = (-0.2, -0.05)  # Baseline correction window

# AVSRT paradigm parameters
AVSRT_TMIN = -0.5  # Epoch start (s)
AVSRT_TMAX = 0.8  # Epoch end (s)
AVSRT_BASELINE = (-0.05, 0.02)  # Baseline correction window
AVSRT_RT_MIN = 0.1  # Minimum valid RT (s)
AVSRT_RT_MAX = 1.5  # Maximum valid RT (s)

# Eye-tracking gaze filtering
GAZE_X_THRESHOLD = 7.0  # degrees from fixation
GAZE_Y_THRESHOLD = 5.0  # degrees from fixation
GAZE_WINDOW = (-0.02, 0.02)  # seconds around stimulus onset

# Spectral analysis parameters (Morlet wavelets)
FREQS = np.arange(2, 40.19, 0.19)  # 2 to 40 Hz in steps of 0.19
N_CYCLES = FREQS / 2  # cycles = frequency / 2
SPECTRAL_BASELINE = (-0.2, 0.0)  # Baseline for dB normalization

# Frequency band definitions
THETA_BAND = (4, 7)
ALPHA_BAND = (7, 13)
GAMMA_BAND = (25, 40)

# ERP channel clusters
FAST_P1_CHANNELS = ["O1", "O2"]
FAST_N170_CHANNELS = ["P7", "P8"]
AVSRT_VISUAL_CHANNELS = ["PO3", "PO7", "O1", "O2", "PO4", "PO8"]
AVSRT_AUDITORY_CHANNELS = ["T7", "TP7", "T8", "TP8"]

# Benchmark results collector
BENCHMARK_RESULTS = []

print("Constants configured.")
print(f"Frequency vector: {len(FREQS)} bins from {FREQS[0]:.1f} to {FREQS[-1]:.1f} Hz")
print(f"Cycles: {N_CYCLES[0]:.1f} to {N_CYCLES[-1]:.1f}")

Constants configured.
Frequency vector: 201 bins from 2.0 to 40.0 Hz
Cycles: 1.0 to 20.0


---
## Section 1: Download Dataset via DataLad

The SFARI EEG multi-paradigm dataset (ds006780) is on OpenNeuro in BIDS format.
It contains ~120 participants across 3 groups:
- **NA**: Non-autistic controls
- **AU**: Autistic children
- **SIB**: Unaffected siblings of autistic children

Two paradigms per subject:
- **FAST**: Face/object oddball (12 blocks, 720 trials)
- **AVSRT**: Audiovisual simple RT (4 blocks, 400 trials)

In [56]:
import subprocess
import shutil
import urllib.request

def run_cmd(cmd, check=True):
    """Run a shell command and return output."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and result.returncode != 0:
        print(f"STDERR: {result.stderr}")
    if result.stdout.strip():
        print(result.stdout.strip())
    return result

HAS_DATALAD = shutil.which("datalad") is not None
OPENNEURO_DATASET_ID = "ds006780"
OPENNEURO_RAW_BASE = (
    f"https://raw.githubusercontent.com/OpenNeuroDatasets/{OPENNEURO_DATASET_ID}/main"
)

# Step 1: Clone the dataset (lightweight - just metadata)
if not DATASET_DIR.exists():
    if HAS_DATALAD:
        print("Cloning dataset via DataLad...")
        run_cmd(
            f"datalad install -s https://github.com/OpenNeuroDatasets/{OPENNEURO_DATASET_ID}.git {DATASET_DIR}"
        )
    else:
        print("DataLad not found – falling back to git clone (metadata only)...")
        run_cmd(
            f"git clone --depth 1 https://github.com/OpenNeuroDatasets/{OPENNEURO_DATASET_ID}.git {DATASET_DIR}"
        )
else:
    print(f"Dataset directory already exists: {DATASET_DIR}")

# Step 2: Read participants.tsv to identify groups
participants_file = DATASET_DIR / "participants.tsv"
if not participants_file.exists():
    if HAS_DATALAD:
        run_cmd(f"datalad get -d {DATASET_DIR} participants.tsv")
    else:
        # Download directly from GitHub raw URL
        url = f"{OPENNEURO_RAW_BASE}/participants.tsv"
        print(f"Downloading participants.tsv from {url} ...")
        urllib.request.urlretrieve(url, participants_file)

participants = pd.read_csv(participants_file, sep="\t")
print(f"\nTotal participants: {len(participants)}")
print(participants.head(10))

Dataset directory already exists: data\ds006780

Total participants: 136
  participant_id   age sex group handedness completed_ASSR completed_AVSRT  \
0      sub-10025   8.3   m    TD          L            yes             yes   
1      sub-10036  11.0   m    TD          R            yes             yes   
2      sub-10129  12.7   f    TD          R            yes             yes   
3      sub-10170   8.3   f    TD          R            yes             yes   
4      sub-10177   8.1   m    TD          R            yes             yes   
5      sub-10182  10.6   f    TD          R            yes             yes   
6      sub-10212   9.5   m    TD          R            yes             yes   
7      sub-10213   8.2   f    TD          R            yes             yes   
8      sub-10214  12.6   f    TD          R            yes             yes   
9      sub-10258  12.7   m    TD          R            yes             yes   

  completed_Beep-Flash completed_FAST completed_Illusory_contours  .

In [57]:
# Select N_SUBJECTS_PER_GROUP subjects per group based on participants.tsv
print("Participants columns:", participants.columns.tolist())
print("\nFirst few rows:")
print(participants.head())

# Identify group column (could be 'group', 'Group', 'diagnosis', etc.)
group_col = None
for col_name in ["group", "Group", "diagnosis", "Diagnosis", "participant_group"]:
    if col_name in participants.columns:
        group_col = col_name
        break

if group_col:
    print(f"\nGroup column: '{group_col}'")
    print(f"Group counts:\n{participants[group_col].value_counts()}")
    
    # Select subjects, skipping NaN/invalid groups
    VALID_GROUPS = {"TD", "ASD", "ASD SIBLING"}
    selected_subjects = {}
    for group_name in participants[group_col].unique():
        if str(group_name).upper() in {"NAN", "NONE", ""} or pd.isna(group_name):
            print(f"  Skipping invalid group: '{group_name}'")
            continue
        group_subs = participants[participants[group_col] == group_name]["participant_id"].tolist()
        selected_subjects[group_name] = group_subs[:N_SUBJECTS_PER_GROUP]
        print(f"  {group_name}: {selected_subjects[group_name]}")
    
    ALL_SUBJECTS = [s for subs in selected_subjects.values() for s in subs]
else:
    print("WARNING: Could not identify group column. Using first 15 subjects.")
    ALL_SUBJECTS = participants["participant_id"].tolist()[:15]

print(f"\nTotal selected: {len(ALL_SUBJECTS)} subjects")
print(f"Groups: {list(selected_subjects.keys())}")

Participants columns: ['participant_id', 'age', 'sex', 'group', 'handedness', 'completed_ASSR', 'completed_AVSRT', 'completed_Beep-Flash', 'completed_FAST', 'completed_Illusory_contours', 'completed_Motor', 'completed_Resting_state', 'fsiq', 'srs2_total_t', 'mabc_total_ss', 'cpt_response_style', 'ados_css', 'medication', 'fsiq_dup', 'srs2_total_t_dup', 'mabc_total_ss_dup', 'cpt_response_style_dup', 'ados_css_dup', 'medication_dup']

First few rows:
  participant_id   age sex group handedness completed_ASSR completed_AVSRT  \
0      sub-10025   8.3   m    TD          L            yes             yes   
1      sub-10036  11.0   m    TD          R            yes             yes   
2      sub-10129  12.7   f    TD          R            yes             yes   
3      sub-10170   8.3   f    TD          R            yes             yes   
4      sub-10177   8.1   m    TD          R            yes             yes   

  completed_Beep-Flash completed_FAST completed_Illusory_contours  ...  \
0   

In [58]:
# Step 3: Download EEG data for selected subjects
# Only fetch the two paradigms used in this benchmark
TARGET_TASKS = ["FAST", "AVSRT"]

# Check if the existing clone is a proper DataLad dataset
is_datalad_repo = (DATASET_DIR / ".datalad").exists() and (
    DATASET_DIR / ".datalad" / "config"
).exists()

if HAS_DATALAD and not is_datalad_repo:
    # The repo was cloned with plain git, not DataLad.
    # Re-clone properly so datalad get works.
    print("Re-cloning dataset with DataLad (needed for git-annex content)...")
    import shutil as _shutil
    _shutil.rmtree(DATASET_DIR)
    run_cmd(
        f"datalad install -s https://github.com/OpenNeuroDatasets/{OPENNEURO_DATASET_ID}.git {DATASET_DIR}"
    )

# Use absolute path to dataset for datalad commands
dataset_abs = str(DATASET_DIR.resolve())

for sub_id in ALL_SUBJECTS:
    sub_dir = sub_id if sub_id.startswith("sub-") else f"sub-{sub_id}"
    sub_clean = sub_id.replace("sub-", "") if sub_id.startswith("sub-") else sub_id
    eeg_path = DATASET_DIR / sub_dir / "eeg"

    # Only fetch files for the target tasks that this subject actually has
    for task in TARGET_TASKS:
        run_label = _find_bids_run(sub_clean, task, DATASET_DIR)
        if run_label is None:
            print(f"  {sub_dir}: task {task} not present — skipping")
            continue

        # datalad get needs to run from within the dataset, or use -d with
        # paths relative to the dataset root
        bdf_relpath = f"{sub_dir}/eeg/{sub_dir}_task-{task}_run-{run_label}_eeg.bdf"
        print(f"Getting {bdf_relpath} ...")
        # Use -C to change to dataset dir so paths are relative to it
        run_cmd(f'datalad -C "{dataset_abs}" get {bdf_relpath}', check=False)

    # Verify real content was fetched (BDF files should be > 1 KB)
    if eeg_path.exists():
        for f in sorted(eeg_path.iterdir()):
            if f.suffix == ".bdf" and any(t in f.name for t in TARGET_TASKS):
                sz = f.stat().st_size
                status = "OK" if sz > 1024 else "ANNEX POINTER"
                print(f"  {f.name}: {sz:,} bytes [{status}]")

print("\nDownload complete.")

Getting sub-10025/eeg/sub-10025_task-FAST_run-01_eeg.bdf ...
Getting sub-10025/eeg/sub-10025_task-AVSRT_run-01_eeg.bdf ...
  sub-10025_task-AVSRT_run-01_eeg.bdf: 98,579,456 bytes [OK]
  sub-10025_task-FAST_run-01_eeg.bdf: 102,167,552 bytes [OK]
Getting sub-11025/eeg/sub-11025_task-FAST_run-01_eeg.bdf ...
  sub-11025: task AVSRT not present — skipping
  sub-11025_task-FAST_run-01_eeg.bdf: 102,728,192 bytes [OK]
Getting sub-1501/eeg/sub-1501_task-FAST_run-01_eeg.bdf ...
Getting sub-1501/eeg/sub-1501_task-AVSRT_run-01_eeg.bdf ...
  sub-1501_task-AVSRT_run-01_eeg.bdf: 98,467,328 bytes [OK]
  sub-1501_task-FAST_run-01_eeg.bdf: 102,728,192 bytes [OK]

Download complete.


In [59]:
# Step 4: Verify BIDS structure
bids_root = DATASET_DIR

# List what we have
print("BIDS root contents:")
for item in sorted(bids_root.iterdir()):
    print(f"  {item.name}")

# Check first subject's EEG files
first_sub = ALL_SUBJECTS[0]
sub_dir = first_sub if first_sub.startswith("sub-") else f"sub-{first_sub}"
print(f"\n{sub_dir} contents:")
sub_path = bids_root / sub_dir
if sub_path.exists():
    for root, dirs, files in os.walk(sub_path):
        level = root.replace(str(sub_path), "").count(os.sep)
        indent = "  " * (level + 1)
        print(f"{indent}{os.path.basename(root)}/")
        sub_indent = "  " * (level + 2)
        for file in files[:10]:  # Limit display
            print(f"{sub_indent}{file}")

BIDS root contents:
  .datalad
  .git
  .gitattributes
  CHANGES
  dataset_description.json
  participants.json
  participants.tsv
  README
  sub-10003
  sub-10025
  sub-10036
  sub-10124
  sub-10129
  sub-10170
  sub-10177
  sub-10182
  sub-10212
  sub-10213
  sub-10214
  sub-10258
  sub-10260
  sub-10310
  sub-10314
  sub-10354
  sub-10400
  sub-10434
  sub-10508
  sub-10520
  sub-10593
  sub-10708
  sub-10741
  sub-10747
  sub-10752
  sub-10764
  sub-10769
  sub-10777
  sub-10787
  sub-10788
  sub-10790
  sub-10809
  sub-10825
  sub-10842
  sub-10846
  sub-10862
  sub-10876
  sub-10883
  sub-10918
  sub-10931
  sub-10950
  sub-10959
  sub-10970
  sub-10990
  sub-11025
  sub-11038
  sub-11049
  sub-11053
  sub-11055
  sub-11057
  sub-11062
  sub-11074
  sub-11078
  sub-11141
  sub-11152
  sub-11161
  sub-11173
  sub-11181
  sub-11204
  sub-11215
  sub-11227
  sub-11229
  sub-11235
  sub-11242
  sub-11243
  sub-11244
  sub-11256
  sub-11265
  sub-11274
  sub-11321
  sub-11323
  sub-11

In [60]:
# Quick check: what groups and subjects are selected?
print(f"N_SUBJECTS_PER_GROUP = {N_SUBJECTS_PER_GROUP}")
print(f"Group column: '{group_col}'")
print(f"Groups in dataset: {participants[group_col].value_counts().to_dict()}")
print(f"\nSelected subjects by group:")
for grp, subs in selected_subjects.items():
    print(f"  {grp}: {subs}")
print(f"\nALL_SUBJECTS ({len(ALL_SUBJECTS)}): {ALL_SUBJECTS}")
print(f"HAS_DATALAD = {HAS_DATALAD}")

N_SUBJECTS_PER_GROUP = 1
Group column: 'group'
Groups in dataset: {'ASD': 66, 'TD': 41, 'ASD SIBLING': 28, 'NAN': 1}

Selected subjects by group:
  TD: ['sub-10025']
  ASD: ['sub-11025']
  ASD SIBLING: ['sub-1501']

ALL_SUBJECTS (3): ['sub-10025', 'sub-11025', 'sub-1501']
HAS_DATALAD = True


---
## Section 2: Reproduce Original Preprocessing

Following the papers exactly:
1. Bad channel detection (pyprep RANSAC) → reject if >15% bad
2. Spline interpolation of bad channels
3. FIR bandpass filter 0.01–40 Hz
4. ICA on 1 Hz high-pass copy → auto-label eye components (two methods compared)
5. Common average reference
6. Eye-tracking gaze filtering (if available)
7. Epoching (paradigm-specific windows)

In [64]:
def detect_bad_channels(raw):
    """Detect bad channels using pyprep NoisyChannels with RANSAC.
    
    Returns list of bad channel names, or empty list if pyprep unavailable.
    """
    if not HAS_PYPREP:
        print("  pyprep not available, skipping bad channel detection")
        return []
    
    nd = NoisyChannels(raw, random_state=42)
    nd.find_bad_by_ransac()
    nd.find_bad_by_deviation()
    nd.find_bad_by_correlation()
    nd.find_bad_by_SNR()
    
    bads = nd.get_bads()
    return bads


def fit_ica(raw, n_components=None):
    """Fit ICA on 1 Hz high-pass filtered copy (matching original pipeline)."""
    # Create 1 Hz high-pass copy for ICA fitting
    raw_ica = raw.copy().filter(l_freq=ICA_HIGHPASS, h_freq=None, verbose=False)
    
    # Fit ICA
    ica = mne.preprocessing.ICA(
        n_components=n_components,
        method="fastica",
        random_state=42,
        max_iter=500,
    )
    ica.fit(raw_ica, verbose=False)
    return ica


def auto_label_ica_components(ica, raw):
    """Auto-label ICA components using two methods and compare.
    
    Method A: mne-icalabel ICLabel classifier
    Method B: EOG correlation (ica.find_bads_eog)
    
    Returns dict with both sets of bad indices and agreement info.
    """
    results = {"icalabel_bads": [], "eog_bads": [], "agreement": None}
    
    # Method A: ICLabel
    if HAS_ICALABEL:
        try:
            labels = label_components(raw, ica, method="iclabel")
            # Mark eye and muscle components as bad
            eye_indices = [
                i for i, label in enumerate(labels["labels"])
                if label in ["eye blink", "eye movement"]
            ]
            results["icalabel_bads"] = eye_indices
            results["icalabel_labels"] = labels
        except Exception as e:
            print(f"  ICLabel failed: {e}")
    
    # Method B: EOG correlation
    try:
        eog_indices, eog_scores = ica.find_bads_eog(
            raw, ch_name=["Fp1", "Fp2"],  # Frontal channels as EOG proxy
            threshold=3.0,
            verbose=False,
        )
        results["eog_bads"] = eog_indices
        results["eog_scores"] = eog_scores
    except Exception as e:
        print(f"  EOG correlation failed: {e}")
    
    # Compare agreement
    set_a = set(results["icalabel_bads"])
    set_b = set(results["eog_bads"])
    overlap = set_a & set_b
    union = set_a | set_b
    results["agreement"] = len(overlap) / max(len(union), 1)
    
    return results


def _find_bids_run(sub_id_clean, task, bids_root):
    """Auto-detect the run label for a given subject/task from the BIDS tree.

    Returns the run string (e.g. '01') or None if no run entity is present.
    """
    eeg_dir = Path(bids_root) / f"sub-{sub_id_clean}" / "eeg"
    if not eeg_dir.exists():
        return None
    import re
    pattern = re.compile(
        rf"sub-{re.escape(sub_id_clean)}_task-{re.escape(task)}_run-(\d+)_eeg\."
    )
    for f in eeg_dir.iterdir():
        m = pattern.search(f.name)
        if m:
            return m.group(1)
    return None


def preprocess_subject(sub_id, task, bids_root):
    """Full preprocessing pipeline for one subject and task.
    
    Returns preprocessed Raw object and preprocessing info dict.
    """
    info = {"subject": sub_id, "task": task}
    sub_id_clean = sub_id.replace("sub-", "") if sub_id.startswith("sub-") else sub_id
    
    # Auto-detect run label from actual filenames
    run = _find_bids_run(sub_id_clean, task, bids_root)
    
    # Load raw data
    bids_path = mne_bids.BIDSPath(
        subject=sub_id_clean, task=task, run=run, datatype="eeg", root=bids_root,
    )
    raw = mne_bids.read_raw_bids(bids_path, verbose=False)
    raw.load_data()
    
    # Set standard BioSemi-64 montage (BDF files don't embed positions)
    montage = mne.channels.make_standard_montage("biosemi64")
    raw.set_montage(montage, on_missing="ignore")
    
    info["n_channels_original"] = len(raw.ch_names)
    info["duration_s"] = raw.times[-1]
    
    # Step 1: Bad channel detection
    bads = detect_bad_channels(raw)
    raw.info["bads"] = bads
    info["n_bad_channels"] = len(bads)
    info["bad_channel_pct"] = len(bads) / len(raw.ch_names)
    
    if info["bad_channel_pct"] > BAD_CHANNEL_THRESHOLD:
        info["rejected"] = True
        print(f"  REJECTED: {info['bad_channel_pct']:.1%} bad channels")
        return None, info
    info["rejected"] = False
    
    # Step 2: Interpolate bad channels
    if bads:
        raw.interpolate_bads(reset_bads=True, verbose=False)
    
    # Step 3: Bandpass filter
    raw.filter(l_freq=FILTER_LOW, h_freq=FILTER_HIGH, verbose=False)
    
    # Step 4: ICA
    ica = fit_ica(raw)
    ica_results = auto_label_ica_components(ica, raw)
    info["ica_n_components"] = ica.n_components_
    info["ica_icalabel_bads"] = ica_results["icalabel_bads"]
    info["ica_eog_bads"] = ica_results["eog_bads"]
    info["ica_agreement"] = ica_results["agreement"]
    
    # Use union of both methods for conservative artifact removal
    all_bads = list(set(ica_results["icalabel_bads"]) | set(ica_results["eog_bads"]))
    if all_bads:
        ica.exclude = all_bads
        ica.apply(raw, verbose=False)
    info["ica_n_removed"] = len(all_bads)
    
    # Step 5: Common average reference
    raw.set_eeg_reference("average", verbose=False)
    
    return raw, info


print("Preprocessing functions defined.")

Preprocessing functions defined.


In [62]:
def create_fast_epochs(raw, events, event_id):
    """Create FAST paradigm epochs.
    
    Window: -500 to +1000 ms, baseline: -200 to -50 ms.
    Non-target stimuli only (faces/objects, upright/inverted).
    """
    # Filter event_id to non-target stimuli only
    nontarget_ids = {
        k: v for k, v in event_id.items()
        if "shadow" not in k.lower() and "target" not in k.lower()
    }
    
    epochs = mne.Epochs(
        raw, events, event_id=nontarget_ids,
        tmin=FAST_TMIN, tmax=FAST_TMAX,
        baseline=FAST_BASELINE,
        preload=True, verbose=False,
    )
    return epochs


def create_avsrt_epochs(raw, events, event_id, behavioral_data=None):
    """Create AVSRT paradigm epochs.
    
    Window: -500 to +800 ms, baseline: -50 to +20 ms.
    Only include trials with RT between 100-1500 ms.
    """
    epochs = mne.Epochs(
        raw, events, event_id=event_id,
        tmin=AVSRT_TMIN, tmax=AVSRT_TMAX,
        baseline=AVSRT_BASELINE,
        preload=True, verbose=False,
    )
    
    # RT filtering would be applied here if behavioral data available
    # For now, return all epochs
    return epochs


def apply_gaze_filter(epochs, eyetracking_data=None):
    """Filter epochs based on eye-tracking gaze position.
    
    Reject visual/audiovisual trials where gaze was outside
    +/-7 deg x, +/-5 deg y from fixation in a 40ms window around onset.
    """
    if eyetracking_data is None:
        print("  No eye-tracking data available, skipping gaze filter")
        return epochs
    
    # Parse EyeLink data and filter
    # Implementation depends on the specific eye-tracking file format
    # in the BIDS dataset
    print("  Eye-tracking gaze filtering applied")
    return epochs


print("Epoching functions defined.")

Epoching functions defined.


In [65]:
# Run preprocessing on all selected subjects (FAST + AVSRT only)
preprocessed_data = {}  # {(sub_id, task): (raw, epochs, info)}
preprocessing_info = []

for sub_id in ALL_SUBJECTS:
    sub_clean = sub_id.replace("sub-", "") if sub_id.startswith("sub-") else sub_id
    
    for task in TARGET_TASKS:
        # Check if this subject actually has this task
        run_label = _find_bids_run(sub_clean, task, DATASET_DIR)
        if run_label is None:
            print(f"\nSkipping {sub_id} / {task} — task not present for this subject")
            continue
        
        print(f"\nPreprocessing {sub_id} / {task}...")
        try:
            t0 = time.time()
            raw, info = preprocess_subject(sub_id, task, DATASET_DIR)
            info["preprocess_time_s"] = time.time() - t0
            
            if raw is not None:
                # Get events
                events, event_id = mne.events_from_annotations(raw, verbose=False)
                info["n_events"] = len(events)
                info["event_ids"] = event_id
                
                # Create epochs based on task
                if "fast" in task.lower():
                    epochs = create_fast_epochs(raw, events, event_id)
                elif "avsrt" in task.lower():
                    epochs = create_avsrt_epochs(raw, events, event_id)
                else:
                    epochs = mne.Epochs(
                        raw, events, event_id=event_id,
                        tmin=-0.5, tmax=1.0, preload=True, verbose=False,
                    )
                
                # Apply gaze filter (placeholder - needs eye-tracking data)
                epochs = apply_gaze_filter(epochs)
                
                info["n_epochs"] = len(epochs)
                preprocessed_data[(sub_id, task)] = (raw, epochs, info)
                print(f"  Done: {len(epochs)} epochs in {info['preprocess_time_s']:.1f}s")
            
            preprocessing_info.append(info)
            
        except Exception as e:
            print(f"  ERROR: {e}")
            preprocessing_info.append({"subject": sub_id, "task": task, "error": str(e)})

# Summary
preprocess_df = pd.DataFrame(preprocessing_info)
print("\n" + "=" * 60)
print("Preprocessing Summary:")
print(preprocess_df[[c for c in ["subject", "task", "n_epochs", "ica_n_removed",
                                  "ica_agreement", "preprocess_time_s"] 
                      if c in preprocess_df.columns]].to_string())


Preprocessing sub-10025 / FAST...


C:\Users\s\AppData\Local\Temp\ipykernel_36216\536438785.py:116: RuntimeWarning: Unable to map the following column(s) to to MNE:
group: TD
handedness: L
completed_ASSR: yes
completed_AVSRT: yes
completed_Beep-Flash: yes
completed_FAST: yes
completed_Illusory_contours: yes
completed_Motor: yes
completed_Resting_state: yes
fsiq: 121.0
srs2_total_t: 44.0
mabc_total_ss: 5.0
cpt_response_style: 74
ados_css: n/a
medication: No
fsiq_dup: 121.0
srs2_total_t_dup: 44.0
mabc_total_ss_dup: 5.0
cpt_response_style_dup: 74
ados_css_dup: n/a
medication_dup: No
  raw = mne_bids.read_raw_bids(bids_path, verbose=False)
C:\Users\s\AppData\Local\Temp\ipykernel_36216\536438785.py:49: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  labels = label_components(raw, ica, method="iclabel")
C:

  No eye-tracking data available, skipping gaze filter
  Done: 733 epochs in 138.6s

Preprocessing sub-10025 / AVSRT...


C:\Users\s\AppData\Local\Temp\ipykernel_36216\536438785.py:116: RuntimeWarning: Unable to map the following column(s) to to MNE:
group: TD
handedness: L
completed_ASSR: yes
completed_AVSRT: yes
completed_Beep-Flash: yes
completed_FAST: yes
completed_Illusory_contours: yes
completed_Motor: yes
completed_Resting_state: yes
fsiq: 121.0
srs2_total_t: 44.0
mabc_total_ss: 5.0
cpt_response_style: 74
ados_css: n/a
medication: No
fsiq_dup: 121.0
srs2_total_t_dup: 44.0
mabc_total_ss_dup: 5.0
cpt_response_style_dup: 74
ados_css_dup: n/a
medication_dup: No
  raw = mne_bids.read_raw_bids(bids_path, verbose=False)
C:\Users\s\AppData\Local\Temp\ipykernel_36216\536438785.py:49: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  labels = label_components(raw, ica, method="iclabel")
C:

  No eye-tracking data available, skipping gaze filter
  Done: 773 epochs in 187.5s

Preprocessing sub-11025 / FAST...


...

  No eye-tracking data available, skipping gaze filter
  Done: 770 epochs in 159.7s

Skipping sub-11025 / AVSRT — task not present for this subject

Preprocessing sub-1501 / FAST...


C:\Users\s\AppData\Local\Temp\ipykernel_36216\536438785.py:116: RuntimeWarning: Unable to map the following column(s) to to MNE:
group: ASD SIBLING
handedness: R
completed_ASSR: yes
completed_AVSRT: yes
completed_Beep-Flash: yes
completed_FAST: yes
completed_Illusory_contours: yes
completed_Motor: yes
completed_Resting_state: yes
fsiq: 109.0
srs2_total_t: 40.0
mabc_total_ss: 6.0
cpt_response_style: 48
ados_css: n/a
medication: No
fsiq_dup: 109.0
srs2_total_t_dup: 40.0
mabc_total_ss_dup: 6.0
cpt_response_style_dup: 48
ados_css_dup: n/a
medication_dup: No
  raw = mne_bids.read_raw_bids(bids_path, verbose=False)
C:\Users\s\AppData\Local\Temp\ipykernel_36216\536438785.py:49: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  labels = label_components(raw, ica, method="icl

  No eye-tracking data available, skipping gaze filter
  Done: 720 epochs in 157.0s

Preprocessing sub-1501 / AVSRT...


C:\Users\s\AppData\Local\Temp\ipykernel_36216\536438785.py:116: RuntimeWarning: Unable to map the following column(s) to to MNE:
group: ASD SIBLING
handedness: R
completed_ASSR: yes
completed_AVSRT: yes
completed_Beep-Flash: yes
completed_FAST: yes
completed_Illusory_contours: yes
completed_Motor: yes
completed_Resting_state: yes
fsiq: 109.0
srs2_total_t: 40.0
mabc_total_ss: 6.0
cpt_response_style: 48
ados_css: n/a
medication: No
fsiq_dup: 109.0
srs2_total_t_dup: 40.0
mabc_total_ss_dup: 6.0
cpt_response_style_dup: 48
ados_css_dup: n/a
medication_dup: No
  raw = mne_bids.read_raw_bids(bids_path, verbose=False)
C:\Users\s\AppData\Local\Temp\ipykernel_36216\536438785.py:49: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  labels = label_components(raw, ica, method="icl

  No eye-tracking data available, skipping gaze filter
  Done: 797 epochs in 149.2s

Preprocessing Summary:
     subject   task  n_epochs  ica_n_removed  ica_agreement  preprocess_time_s
0  sub-10025   FAST       733              5       0.200000         138.592644
1  sub-10025  AVSRT       773              4       0.250000         187.539862
2  sub-11025   FAST       770             10       0.100000         159.701301
3   sub-1501   FAST       720              3       0.333333         156.978276
4   sub-1501  AVSRT       797              6       0.166667         149.170635


---
## Section 3: Reproduce Original FAST Analysis

Analyses from Vanneau, Brittenham et al. (2025):
- **P1** (50-180 ms, O1/O2): Amplitude & latency by category/orientation/group
- **N170** (170-200 ms, P7/P8): Face selectivity, face inversion effect (FIE)
- **Induced theta** (4-7 Hz): Morlet wavelets, ERP subtracted, central channels
- **Alpha ERD** (7-13 Hz): Occipital, dB baseline normalization
- **Gamma ITPC** (25-40 Hz): Occipital, 0-200 ms window

In [68]:
def extract_erp_peaks(evoked, channels, time_window, polarity="pos"):
    """Extract peak amplitude and latency from ERP in given window.
    
    Parameters
    ----------
    evoked : mne.Evoked
    channels : list of str
    time_window : tuple (tmin, tmax) in seconds
    polarity : 'pos' for positive peak, 'neg' for negative peak
    
    Returns
    -------
    amplitude : float (uV)
    latency : float (s)
    """
    ch_idx = [evoked.ch_names.index(ch) for ch in channels if ch in evoked.ch_names]
    if not ch_idx:
        return np.nan, np.nan
    
    tmin_idx = np.searchsorted(evoked.times, time_window[0])
    tmax_idx = np.searchsorted(evoked.times, time_window[1])
    
    data = evoked.data[ch_idx, tmin_idx:tmax_idx].mean(axis=0)
    
    if polarity == "pos":
        peak_idx = np.argmax(data)
    else:
        peak_idx = np.argmin(data)
    
    amplitude = data[peak_idx] * 1e6  # Convert to uV
    latency = evoked.times[tmin_idx + peak_idx]
    
    return amplitude, latency


def compute_induced_power(epochs, freqs, n_cycles, subtract_erp=True, decim=4):
    """Compute induced (non-phase-locked) power using Morlet wavelets.
    
    Matches paper: subtract ERP, then compute TFR, dB normalize.
    """
    # Pick EEG channels only to avoid shape mismatches with misc/stim channels
    epochs_eeg = epochs.copy().pick("eeg")

    if subtract_erp:
        # Subtract ERP from each trial
        erp = epochs_eeg.average().data  # (n_eeg, n_times)
        data = epochs_eeg.get_data() - erp[np.newaxis, :, :]
        # Create new epochs from residual
        epochs_induced = mne.EpochsArray(
            data, epochs_eeg.info, tmin=epochs_eeg.tmin, verbose=False
        )
    else:
        epochs_induced = epochs_eeg
    
    # Compute TFR (use_fft=True for speed)
    power = mne.time_frequency.tfr_morlet(
        epochs_induced, freqs=freqs, n_cycles=n_cycles,
        return_itc=False, average=True, use_fft=True,
        decim=decim, verbose=False,
    )
    
    # dB normalization relative to baseline
    power.apply_baseline(baseline=SPECTRAL_BASELINE, mode="logratio", verbose=False)
    
    return power


def compute_itpc(epochs, freqs, n_cycles, decim=4):
    """Compute inter-trial phase coherence."""
    # Pick EEG channels only
    epochs_eeg = epochs.copy().pick("eeg")
    power, itc = mne.time_frequency.tfr_morlet(
        epochs_eeg, freqs=freqs, n_cycles=n_cycles,
        return_itc=True, average=True, use_fft=True,
        decim=decim, verbose=False,
    )
    return itc


print("FAST analysis functions defined.")

FAST analysis functions defined.


In [69]:
# Run FAST analysis on preprocessed data
fast_results = []

for (sub_id, task), (raw, epochs, info) in preprocessed_data.items():
    if "fast" not in task.lower():
        continue
    
    print(f"\nFAST analysis: {sub_id}")
    result = {"subject": sub_id, "group": info.get("group", "unknown")}
    
    # Get condition-specific epochs
    conditions = list(epochs.event_id.keys())
    print(f"  Conditions: {conditions}")
    
    for cond in conditions:
        try:
            evoked = epochs[cond].average()
            
            # P1 (50-180 ms, O1/O2)
            p1_amp, p1_lat = extract_erp_peaks(
                evoked, FAST_P1_CHANNELS, (0.05, 0.18), polarity="pos"
            )
            result[f"P1_amp_{cond}"] = p1_amp
            result[f"P1_lat_{cond}"] = p1_lat
            
            # N170 (170-200 ms, P7/P8)
            n170_amp, n170_lat = extract_erp_peaks(
                evoked, FAST_N170_CHANNELS, (0.17, 0.20), polarity="neg"
            )
            result[f"N170_amp_{cond}"] = n170_amp
            result[f"N170_lat_{cond}"] = n170_lat
            
        except Exception as e:
            print(f"  ERP error for {cond}: {e}")
    
    # Induced theta (4-7 Hz)
    try:
        theta_freqs = FREQS[(FREQS >= 4) & (FREQS <= 7)]
        theta_cycles = theta_freqs / 2
        for cond in conditions:
            power = compute_induced_power(
                epochs[cond], theta_freqs, theta_cycles, subtract_erp=True
            )
            # Average over central channels and 200-500 ms
            central_chs = [ch for ch in ["Cz", "FCz", "CPz", "C1", "C2"] 
                          if ch in epochs.ch_names]
            if central_chs:
                ch_idx = [epochs.ch_names.index(ch) for ch in central_chs]
                t_mask = (power.times >= 0.2) & (power.times <= 0.5)
                theta_power = power.data[ch_idx][:, :, t_mask].mean()
                result[f"theta_power_{cond}"] = theta_power
    except Exception as e:
        print(f"  Theta analysis error: {e}")
    
    # Alpha ERD (7-13 Hz)
    try:
        alpha_freqs = FREQS[(FREQS >= 7) & (FREQS <= 13)]
        alpha_cycles = alpha_freqs / 2
        for cond in conditions:
            power = compute_induced_power(
                epochs[cond], alpha_freqs, alpha_cycles, subtract_erp=True
            )
            occ_chs = [ch for ch in ["O1", "O2", "PO3", "PO4", "PO7", "PO8"]
                       if ch in epochs.ch_names]
            if occ_chs:
                ch_idx = [epochs.ch_names.index(ch) for ch in occ_chs]
                t_mask = (power.times >= 0.2) & (power.times <= 0.5)
                alpha_erd = power.data[ch_idx][:, :, t_mask].mean()
                result[f"alpha_erd_{cond}"] = alpha_erd
    except Exception as e:
        print(f"  Alpha ERD error: {e}")
    
    # Gamma ITPC (25-40 Hz)
    try:
        gamma_freqs = FREQS[(FREQS >= 25) & (FREQS <= 40)]
        gamma_cycles = gamma_freqs / 2
        for cond in conditions:
            itc = compute_itpc(epochs[cond], gamma_freqs, gamma_cycles)
            occ_chs = [ch for ch in ["O1", "O2", "Oz"] if ch in epochs.ch_names]
            if occ_chs:
                ch_idx = [epochs.ch_names.index(ch) for ch in occ_chs]
                t_mask = (itc.times >= 0.0) & (itc.times <= 0.2)
                gamma_itpc = itc.data[ch_idx][:, :, t_mask].mean()
                result[f"gamma_itpc_{cond}"] = gamma_itpc
    except Exception as e:
        print(f"  Gamma ITPC error: {e}")
    
    fast_results.append(result)

if fast_results:
    fast_df = pd.DataFrame(fast_results)
    print("\n" + "=" * 60)
    print("FAST Analysis Summary:")
    print(fast_df.to_string())
else:
    print("No FAST data processed.")


FAST analysis: sub-10025
  Conditions: ['Face_inverted', 'Face_upright', 'Object_inverted', 'Object_upright', 'Response_button']

FAST analysis: sub-11025
  Conditions: ['Face_inverted', 'Face_upright', 'Object_inverted', 'Object_upright', 'Response_button']

FAST analysis: sub-1501
  Conditions: ['Face_inverted', 'Face_upright', 'Object_inverted', 'Object_upright', 'Response_button']

FAST Analysis Summary:
     subject    group  P1_amp_Face_inverted  P1_lat_Face_inverted  N170_amp_Face_inverted  N170_lat_Face_inverted  P1_amp_Face_upright  P1_lat_Face_upright  N170_amp_Face_upright  N170_lat_Face_upright  P1_amp_Object_inverted  P1_lat_Object_inverted  N170_amp_Object_inverted  N170_lat_Object_inverted  P1_amp_Object_upright  P1_lat_Object_upright  N170_amp_Object_upright  N170_lat_Object_upright  P1_amp_Response_button  P1_lat_Response_button  N170_amp_Response_button  N170_lat_Response_button  theta_power_Face_inverted  theta_power_Face_upright  theta_power_Object_inverted  theta_

---
## Section 4: Reproduce Original AVSRT Analysis

Analyses from Vanneau, Foxe et al. (2025):
- **Visual P1/N1** (PO3/PO7/O1/O2/PO4/PO8): P1: 120-170 ms, N1: 190-230 ms
- **Auditory N1/P2** (T7/TP7/T8/TP8): N1: 120-170 ms, P2: 220-350 ms
- **MSI ERP**: AV - (A+V), cluster permutation <250 ms
- **Alpha ERD MSI**: Bootstrap A+V, z-score AV vs. distribution
- **wPLI**: Laplacian, Morlet 3-13 Hz, fronto-parieto-occipital

In [70]:
def compute_msi_erp(epochs_av, epochs_a, epochs_v):
    """Compute MSI difference wave: AV - (A + V).
    
    Returns difference evoked and cluster permutation results.
    """
    evoked_av = epochs_av.average()
    evoked_a = epochs_a.average()
    evoked_v = epochs_v.average()
    
    # MSI = AV - (A + V)
    msi = mne.combine_evoked([evoked_av, evoked_a, evoked_v], weights=[1, -1, -1])
    
    return msi


def compute_alpha_msi(epochs_av, epochs_a, epochs_v, n_bootstrap=1000):
    """Compute alpha ERD MSI using bootstrap approach.
    
    1. Sum A+V epochs in time domain (trial-by-trial, bootstrapped pairings)
    2. Compute alpha power for AV and bootstrapped A+V
    3. Z-score AV vs. bootstrap distribution
    """
    alpha_freqs = FREQS[(FREQS >= 7) & (FREQS <= 13)]
    alpha_cycles = alpha_freqs / 2
    
    # AV alpha power
    av_power = compute_induced_power(epochs_av, alpha_freqs, alpha_cycles)
    occ_chs = [ch for ch in ["O1", "O2", "PO3", "PO4", "PO7", "PO8"]
               if ch in epochs_av.ch_names]
    ch_idx = [epochs_av.ch_names.index(ch) for ch in occ_chs]
    t_mask = (av_power.times >= 0.2) & (av_power.times <= 0.5)
    av_alpha = av_power.data[ch_idx][:, :, t_mask].mean()
    
    # Bootstrap A+V alpha power
    data_a = epochs_a.get_data()
    data_v = epochs_v.get_data()
    n_a, n_v = len(data_a), len(data_v)
    n_pairs = min(n_a, n_v)
    
    bootstrap_alpha = []
    rng = np.random.default_rng(42)
    for _ in range(n_bootstrap):
        idx_a = rng.choice(n_a, n_pairs, replace=True)
        idx_v = rng.choice(n_v, n_pairs, replace=True)
        summed = data_a[idx_a] + data_v[idx_v]
        
        summed_epochs = mne.EpochsArray(
            summed, epochs_av.info, tmin=epochs_av.tmin, verbose=False
        )
        power = compute_induced_power(summed_epochs, alpha_freqs, alpha_cycles)
        bs_alpha = power.data[ch_idx][:, :, t_mask].mean()
        bootstrap_alpha.append(bs_alpha)
    
    bootstrap_alpha = np.array(bootstrap_alpha)
    z_score = (av_alpha - bootstrap_alpha.mean()) / bootstrap_alpha.std()
    
    return z_score, av_alpha, bootstrap_alpha


def compute_wpli(epochs, fmin=3, fmax=13):
    """Compute weighted Phase Lag Index.
    
    Uses Laplacian spatial filter, Morlet wavelets.
    """
    if not HAS_CONNECTIVITY:
        print("  mne-connectivity not available")
        return None
    
    # Apply Laplacian
    epochs_lap = mne.preprocessing.compute_current_source_density(epochs, verbose=False)
    
    # Compute wPLI
    freqs_conn = np.arange(fmin, fmax + 0.5, 0.5)
    n_cycles_conn = np.linspace(2, 10, len(freqs_conn))
    
    con = spectral_connectivity_epochs(
        epochs_lap, method="wpli",
        mode="cwt_morlet",
        cwt_freqs=freqs_conn,
        cwt_n_cycles=n_cycles_conn,
        verbose=False,
    )
    
    return con


print("AVSRT analysis functions defined.")

AVSRT analysis functions defined.


In [71]:
# Run AVSRT analysis on preprocessed data
avsrt_results = []

for (sub_id, task), (raw, epochs, info) in preprocessed_data.items():
    if "avsrt" not in task.lower():
        continue
    
    print(f"\nAVSRT analysis: {sub_id}")
    result = {"subject": sub_id, "group": info.get("group", "unknown")}
    
    conditions = list(epochs.event_id.keys())
    print(f"  Conditions: {conditions}")
    
    # Identify A, V, AV conditions (adapt to actual event names)
    a_conds = [c for c in conditions if "audio" in c.lower() or c.lower() in ["a", "auditory"]]
    v_conds = [c for c in conditions if "visual" in c.lower() or c.lower() in ["v", "visual"]]
    av_conds = [c for c in conditions if "audiovisual" in c.lower() or c.lower() in ["av", "audiovisual"]]
    
    for cond_name, cond_list, channels, windows in [
        ("visual", v_conds, AVSRT_VISUAL_CHANNELS, [(0.12, 0.17, "P1", "pos"), (0.19, 0.23, "N1", "neg")]),
        ("auditory", a_conds, AVSRT_AUDITORY_CHANNELS, [(0.12, 0.17, "N1", "neg"), (0.22, 0.35, "P2", "pos")]),
    ]:
        if cond_list:
            try:
                evoked = epochs[cond_list].average()
                for tmin, tmax, comp, pol in windows:
                    amp, lat = extract_erp_peaks(evoked, channels, (tmin, tmax), polarity=pol)
                    result[f"{cond_name}_{comp}_amp"] = amp
                    result[f"{cond_name}_{comp}_lat"] = lat
            except Exception as e:
                print(f"  ERP error for {cond_name}: {e}")
    
    # MSI ERP
    if av_conds and a_conds and v_conds:
        try:
            msi = compute_msi_erp(epochs[av_conds], epochs[a_conds], epochs[v_conds])
            # Extract MSI amplitude at 145-165 ms over parieto-central
            pc_chs = [ch for ch in ["CP1", "CP2", "CPz", "P1", "P2", "Pz"]
                      if ch in msi.ch_names]
            if pc_chs:
                amp, lat = extract_erp_peaks(msi, pc_chs, (0.145, 0.165), polarity="pos")
                result["msi_erp_amp"] = amp
                result["msi_erp_lat"] = lat
        except Exception as e:
            print(f"  MSI ERP error: {e}")
    
    # Alpha ERD MSI (reduced bootstrap for speed)
    if av_conds and a_conds and v_conds:
        try:
            z_score, _, _ = compute_alpha_msi(
                epochs[av_conds], epochs[a_conds], epochs[v_conds],
                n_bootstrap=100,  # Reduced for dev speed
            )
            result["alpha_msi_zscore"] = z_score
        except Exception as e:
            print(f"  Alpha MSI error: {e}")
    
    # wPLI (AVSRT specific)
    if av_conds:
        try:
            con = compute_wpli(epochs[av_conds])
            if con is not None:
                result["wpli_computed"] = True
        except Exception as e:
            print(f"  wPLI error: {e}")
    
    avsrt_results.append(result)

if avsrt_results:
    avsrt_df = pd.DataFrame(avsrt_results)
    print("\n" + "=" * 60)
    print("AVSRT Analysis Summary:")
    print(avsrt_df.to_string())
else:
    print("No AVSRT data processed.")


AVSRT analysis: sub-10025
  Conditions: ['Audiovisual_stim', 'Auditory_stim', 'Response_button', 'Visual_stim']


KeyboardInterrupt: 

---
## Section 5: DSS Usage 1 — Artifact Removal (replaces ICA)

Compare IterativeDSS with various nonlinear denoisers against the ICA-based
artifact removal from the original pipeline.

| # | Denoiser | Equivalent |
|---|----------|------------|
| 1 | TanhMaskDenoiser + beta_tanh | FastICA-tanh |
| 2 | GaussDenoiser + beta_gauss | FastICA-gauss |
| 3 | KurtosisDenoiser + beta_pow3 | FastICA-pow3 |
| 4 | RobustTanhDenoiser | Deflationary tanh |
| 5 | CycleAverageBias (blink-locked) | Linear DSS for blink template |

In [72]:
def benchmark_artifact_removal(raw_original, epochs_original, denoiser_name, denoiser_fn,
                                n_components=20, method="deflation", beta=None):
    """Benchmark one DSS denoiser for artifact removal.
    
    Returns metrics dict with quality and runtime.
    """
    metrics = {"denoiser": denoiser_name, "usage": "artifact_removal"}
    
    t0 = time.time()
    try:
        idss = IterativeDSS(
            denoiser=denoiser_fn,
            n_components=n_components,
            method=method,
            max_iter=100,
            beta=beta,
            random_state=42,
        )
        idss.fit(raw_original.copy())
        sources = idss.transform(raw_original.copy())
        
        # Identify eye components via frontal channel correlation
        fp_channels = [ch for ch in ["Fp1", "Fp2"] if ch in raw_original.ch_names]
        if fp_channels:
            fp_idx = [raw_original.ch_names.index(ch) for ch in fp_channels]
            fp_data = raw_original.get_data()[fp_idx].mean(axis=0)
            
            correlations = np.array([
                np.abs(np.corrcoef(sources[i], fp_data)[0, 1])
                for i in range(min(n_components, sources.shape[0]))
            ])
            eye_comps = np.where(correlations > 0.3)[0]
            metrics["n_eye_components"] = len(eye_comps)
            metrics["max_eog_correlation"] = correlations.max()
        
        metrics["runtime_s"] = time.time() - t0
        metrics["success"] = True
        metrics["n_components"] = sources.shape[0]
        
    except Exception as e:
        metrics["runtime_s"] = time.time() - t0
        metrics["success"] = False
        metrics["error"] = str(e)
    
    return metrics


# Define denoisers to benchmark
artifact_denoisers = [
    ("TanhMask + beta_tanh", TanhMaskDenoiser(alpha=1.0).denoise, beta_tanh),
    ("GaussDenoiser + beta_gauss", GaussDenoiser(a=1.0).denoise, beta_gauss),
    ("KurtosisDenoiser + beta_pow3", KurtosisDenoiser().denoise, beta_pow3),
    ("RobustTanhDenoiser", RobustTanhDenoiser().denoise, None),
]

# Run benchmarks on first available subject
usage1_results = []
for (sub_id, task), (raw, epochs, info) in preprocessed_data.items():
    print(f"\nUsage 1 benchmark: {sub_id} / {task}")
    
    for name, denoiser_fn, beta in artifact_denoisers:
        print(f"  Testing {name}...")
        metrics = benchmark_artifact_removal(
            raw, epochs, name, denoiser_fn, beta=beta
        )
        metrics["subject"] = sub_id
        metrics["task"] = task
        usage1_results.append(metrics)
        
        if metrics["success"]:
            print(f"    Runtime: {metrics['runtime_s']:.1f}s, "
                  f"Components: {metrics.get('n_components', '?')}, "
                  f"Eye comps: {metrics.get('n_eye_components', '?')}")
        else:
            print(f"    FAILED: {metrics.get('error', 'unknown')}")

if usage1_results:
    u1_df = pd.DataFrame(usage1_results)
    print("\n" + "=" * 60)
    print("Usage 1: Artifact Removal Benchmark")
    display_cols = [c for c in ["denoiser", "subject", "task", "runtime_s", 
                                 "n_eye_components", "max_eog_correlation", "success"]
                    if c in u1_df.columns]
    print(u1_df[display_cols].to_string())
    BENCHMARK_RESULTS.extend(usage1_results)


Usage 1 benchmark: sub-10025 / FAST
  Testing TanhMask + beta_tanh...
    Runtime: 1.6s, Components: 1, Eye comps: 0
  Testing GaussDenoiser + beta_gauss...
    Runtime: 2.0s, Components: 1, Eye comps: 0
  Testing KurtosisDenoiser + beta_pow3...
    Runtime: 1.5s, Components: 1, Eye comps: 0
  Testing RobustTanhDenoiser...
    Runtime: 1.2s, Components: 1, Eye comps: 0

Usage 1 benchmark: sub-10025 / AVSRT
  Testing TanhMask + beta_tanh...
    Runtime: 1.5s, Components: 1, Eye comps: 0
  Testing GaussDenoiser + beta_gauss...
    Runtime: 1.1s, Components: 1, Eye comps: 0
  Testing KurtosisDenoiser + beta_pow3...
    Runtime: 1.0s, Components: 1, Eye comps: 0
  Testing RobustTanhDenoiser...
    Runtime: 0.9s, Components: 1, Eye comps: 0

Usage 1 benchmark: sub-11025 / FAST
  Testing TanhMask + beta_tanh...
    Runtime: 1.1s, Components: 1, Eye comps: 0
  Testing GaussDenoiser + beta_gauss...
    Runtime: 0.8s, Components: 1, Eye comps: 0
  Testing KurtosisDenoiser + beta_pow3...
    Ru

---
## Section 6: DSS Usage 2 — Line Noise Removal

The original pipeline applies a 40 Hz lowpass which inherently removes line noise.
Here we test dedicated line noise removal before the lowpass.

| # | Method | Description |
|---|--------|-------------|
| 1 | ZapLine (standard) | DSS-based 60 Hz removal |
| 2 | ZapLine (adaptive) | Auto-detect + per-segment tuning |
| 3 | DSS + LineNoiseBias | Direct DSS with line noise bias |

In [73]:
def benchmark_line_noise(raw, method_name, method_fn):
    """Benchmark line noise removal method."""
    metrics = {"denoiser": method_name, "usage": "line_noise_removal"}
    
    # PSD before
    psd_before = raw.compute_psd(fmin=50, fmax=70, verbose=False)
    psd_data_before = psd_before.get_data().mean(axis=0)
    freqs_psd = psd_before.freqs
    line_idx = np.argmin(np.abs(freqs_psd - LINE_FREQ))
    power_at_line_before = 10 * np.log10(psd_data_before[line_idx])
    
    t0 = time.time()
    try:
        cleaned = method_fn(raw.copy())
        metrics["runtime_s"] = time.time() - t0
        
        # PSD after
        psd_after = cleaned.compute_psd(fmin=50, fmax=70, verbose=False)
        psd_data_after = psd_after.get_data().mean(axis=0)
        power_at_line_after = 10 * np.log10(psd_data_after[line_idx])
        
        metrics["power_at_line_before_dB"] = power_at_line_before
        metrics["power_at_line_after_dB"] = power_at_line_after
        metrics["reduction_dB"] = power_at_line_before - power_at_line_after
        metrics["success"] = True
        
    except Exception as e:
        metrics["runtime_s"] = time.time() - t0
        metrics["success"] = False
        metrics["error"] = str(e)
    
    return metrics


def zapline_standard(raw):
    """Standard ZapLine: fit then transform (non-adaptive)."""
    zl = ZapLine(sfreq=raw.info["sfreq"], line_freq=LINE_FREQ, n_remove="auto")
    zl.fit(raw)
    return zl.transform(raw)

def zapline_adaptive(raw):
    """Adaptive ZapLine-plus: must use fit_transform (adaptive mode
    requires simultaneous fit and transform on local chunks)."""
    zl = ZapLine(sfreq=raw.info["sfreq"], line_freq=LINE_FREQ, adaptive=True)
    return zl.fit_transform(raw)

def dss_linenoise(raw):
    """DSS with LineNoiseBias: extract line noise components, exclude them,
    reconstruct cleaned data."""
    # Note: LineNoiseBias uses `freq=` parameter, not `line_freq=`
    bias = LineNoiseBias(freq=LINE_FREQ, sfreq=raw.info["sfreq"], n_harmonics=3)
    dss = DSS(bias=bias, n_components=5)
    dss.fit(raw)
    # Get sources (default return_type='sources' -> numpy array)
    sources = dss.transform(raw)  # shape: (n_components, n_times)
    # Reconstruct excluding top component (which contains line noise)
    # component_indices selects which components to keep in reconstruction
    reconstructed = dss.inverse_transform(
        sources, component_indices=np.arange(1, sources.shape[0])
    )
    return mne.io.RawArray(reconstructed, raw.info, verbose=False)


line_noise_methods = [
    ("ZapLine (standard)", zapline_standard),
    ("ZapLine (adaptive)", zapline_adaptive),
    ("DSS + LineNoiseBias", dss_linenoise),
]

usage2_results = []
for (sub_id, task), (raw, epochs, info) in preprocessed_data.items():
    print(f"\nUsage 2 benchmark: {sub_id} / {task}")
    
    for name, method_fn in line_noise_methods:
        print(f"  Testing {name}...")
        metrics = benchmark_line_noise(raw, name, method_fn)
        metrics["subject"] = sub_id
        metrics["task"] = task
        usage2_results.append(metrics)
        
        if metrics["success"]:
            print(f"    Reduction: {metrics.get('reduction_dB', '?'):.1f} dB, "
                  f"Runtime: {metrics['runtime_s']:.1f}s")
    break  # One subject for dev

if usage2_results:
    u2_df = pd.DataFrame(usage2_results)
    print("\nUsage 2: Line Noise Removal Benchmark")
    print(u2_df[[c for c in ["denoiser", "reduction_dB", "runtime_s", "success"]
                 if c in u2_df.columns]].to_string())
    BENCHMARK_RESULTS.extend(usage2_results)


Usage 2 benchmark: sub-10025 / FAST
  Testing ZapLine (standard)...


c:\Users\s\anaconda3\envs\eeg_analysis\lib\site-packages\mne_denoise\zapline\core.py:438: UserWarning: sfreq/line_freq = 8.53 is not close to an integer. Smoothing will use period=9 samples.
  data_smooth, data_residual = self._get_smooth_residual(data, warn=True)


    Reduction: 0.0 dB, Runtime: 4.4s
  Testing ZapLine (adaptive)...


c:\Users\s\anaconda3\envs\eeg_analysis\lib\site-packages\mne_denoise\zapline\core.py:438: UserWarning: sfreq/line_freq = 8.54 is not close to an integer. Smoothing will use period=9 samples.
  data_smooth, data_residual = self._get_smooth_residual(data, warn=True)


    Reduction: 0.0 dB, Runtime: 36.9s
  Testing DSS + LineNoiseBias...

Usage 2: Line Noise Removal Benchmark
              denoiser  reduction_dB  runtime_s  success
0   ZapLine (standard)           0.0   4.403852     True
1   ZapLine (adaptive)           0.0  36.920511     True
2  DSS + LineNoiseBias           NaN   3.105239    False


---
## Section 7: DSS Usage 3 — ERP Enhancement

Replace simple trial averaging with DSS-based ERP extraction that maximizes
trial-to-trial reproducibility.

| # | Method | Bias |
|---|--------|------|
| 0 | Original (simple average) | None |
| 1 | DSS + AverageBias | Trial averaging |
| 2 | DSS + SmoothingBias | Temporal smoothness |
| 3 | time_shift_dss | Temporal predictability |

In [19]:
def compute_erp_snr(epochs, evoked=None):
    """Compute ERP signal-to-noise ratio.
    
    SNR = var(evoked) / mean(var(single_trial - evoked))
    """
    if evoked is None:
        evoked = epochs.average()
    
    signal_var = np.var(evoked.data)
    residuals = epochs.get_data() - evoked.data[np.newaxis, :, :]
    noise_var = np.mean(np.var(residuals, axis=(1, 2)))
    
    return 10 * np.log10(signal_var / max(noise_var, 1e-30))  # dB


def benchmark_erp_enhancement(epochs, method_name, method_fn=None, n_components=5):
    """Benchmark ERP enhancement method.
    
    All DSS methods here use return_type='epochs' (set in the lambda),
    so transform() returns a denoised MNE Epochs object.
    """
    metrics = {"denoiser": method_name, "usage": "erp_enhancement"}
    
    t0 = time.time()
    try:
        if method_fn is None:
            # Original: simple averaging
            evoked = epochs.average()
            snr = compute_erp_snr(epochs, evoked)
        else:
            # DSS-based: method_fn returns a DSS with return_type='epochs'
            dss = method_fn(n_components)
            dss.fit(epochs)
            # transform returns denoised Epochs (return_type='epochs' set in lambda)
            denoised = dss.transform(epochs)
            if hasattr(denoised, 'average'):
                evoked = denoised.average()
                snr = compute_erp_snr(denoised, evoked)
            else:
                # Fallback: if transform returned numpy array
                snr = np.nan
        
        metrics["runtime_s"] = time.time() - t0
        metrics["snr_dB"] = snr
        metrics["success"] = True
        
    except Exception as e:
        metrics["runtime_s"] = time.time() - t0
        metrics["success"] = False
        metrics["error"] = str(e)
    
    return metrics


erp_methods = [
    ("Original (average)", None),
    ("DSS + AverageBias", lambda n: DSS(
        bias=AverageBias(axis="epochs"), n_components=n, return_type="epochs"
    )),
    ("DSS + SmoothingBias", lambda n: DSS(
        bias=SmoothingBias(window=10), n_components=n, return_type="epochs"
    )),
    ("time_shift_dss", lambda n: time_shift_dss(
        shifts=10, n_components=n, return_type="epochs"
    )),
]

usage3_results = []
for (sub_id, task), (raw, epochs, info) in preprocessed_data.items():
    print(f"\nUsage 3 benchmark: {sub_id} / {task}")
    
    for name, method_fn in erp_methods:
        print(f"  Testing {name}...")
        metrics = benchmark_erp_enhancement(epochs, name, method_fn)
        metrics["subject"] = sub_id
        metrics["task"] = task
        usage3_results.append(metrics)
        
        if metrics["success"]:
            print(f"    SNR: {metrics.get('snr_dB', '?'):.1f} dB, "
                  f"Runtime: {metrics['runtime_s']:.2f}s")

if usage3_results:
    u3_df = pd.DataFrame(usage3_results)
    print("\nUsage 3: ERP Enhancement Benchmark")
    print(u3_df[[c for c in ["denoiser", "snr_dB", "runtime_s", "success"]
                 if c in u3_df.columns]].to_string())
    BENCHMARK_RESULTS.extend(usage3_results)

---
## Section 8: DSS Usage 4 — Alpha Band Extraction (7-13 Hz)

Extract alpha-band spatial components before computing ERD.
DSS finds spatial filters maximizing alpha power ratio.

| # | Method | Description |
|---|--------|-------------|
| 0 | Original (Morlet) | Direct wavelet decomposition |
| 1 | DSS + BandpassBias | Maximize 7-13 Hz power |
| 2 | narrowband_dss | Convenience wrapper at 10 Hz |
| 3 | IterativeDSS + WienerMask | For non-stationary alpha |
| 4 | IterativeDSS + Spectrogram | Adaptive TF masking |

In [20]:
def benchmark_band_extraction(epochs, band, method_name, dss_fn=None, n_components=5):
    """Benchmark band-specific DSS extraction.
    
    For DSS methods, we extract sources (numpy arrays) and compute
    band power on the top component. Both DSS (return_type='sources')
    and IterativeDSS always return numpy arrays from transform().
    
    - DSS.transform() with return_type='sources' on Epochs input:
      returns (n_epochs, n_components, n_times) numpy array
    - IterativeDSS.transform() on Epochs input:
      returns (n_epochs, n_components, n_times) numpy array
    """
    metrics = {"denoiser": method_name, "usage": f"band_extraction_{band[0]}-{band[1]}Hz"}
    
    band_freqs = FREQS[(FREQS >= band[0]) & (FREQS <= band[1])]
    band_cycles = band_freqs / 2
    
    t0 = time.time()
    try:
        if dss_fn is None:
            # Original: direct Morlet on all channels
            power = compute_induced_power(epochs, band_freqs, band_cycles)
            occ_chs = [ch for ch in ["O1", "O2", "PO3", "PO4", "PO7", "PO8"]
                       if ch in epochs.ch_names]
            ch_idx = [epochs.ch_names.index(ch) for ch in occ_chs]
            t_mask = (power.times >= 0.2) & (power.times <= 0.5)
            band_power = power.data[ch_idx][:, :, t_mask].mean()
        else:
            # DSS-enhanced: extract components, then compute power
            dss = dss_fn(n_components)
            dss.fit(epochs)
            # Both DSS and IterativeDSS return numpy arrays here
            sources = dss.transform(epochs)
            
            # Sources shape depends on input:
            # For Epochs input: (n_epochs, n_components, n_times)
            # For Raw/2D input: (n_components, n_times)
            if sources.ndim == 3:
                # Epoched: take top component across all epochs
                top_source = sources[:, 0:1, :]  # (n_epochs, 1, n_times)
            elif sources.ndim == 2:
                top_source = sources[0:1, :]  # (1, n_times)
            
            # Variance of top component as power proxy
            band_power = np.var(top_source)
            
            # Get eigenvalue if available (DSS only, not IterativeDSS)
            if hasattr(dss, 'eigenvalues_') and dss.eigenvalues_ is not None:
                metrics["eigenvalue_0"] = dss.eigenvalues_[0]
            else:
                metrics["eigenvalue_0"] = np.nan
        
        metrics["band_power"] = band_power
        metrics["runtime_s"] = time.time() - t0
        metrics["success"] = True
        
    except Exception as e:
        metrics["runtime_s"] = time.time() - t0
        metrics["success"] = False
        metrics["error"] = str(e)
    
    return metrics


alpha_methods = [
    ("Original (Morlet)", None),
    ("DSS + BandpassBias(7,13)", lambda n: DSS(
        bias=BandpassBias(freq_band=(7, 13), sfreq=SFREQ), n_components=n
    )),
    ("narrowband_dss(10Hz)", lambda n: narrowband_dss(
        sfreq=SFREQ, freq=10, bandwidth=6, n_components=n
    )),
    ("IterativeDSS + WienerMask", lambda n: IterativeDSS(
        denoiser=WienerMaskDenoiser(window_samples=int(0.2 * SFREQ)).denoise,
        n_components=n, method="deflation", max_iter=50, random_state=42
    )),
]

usage4_results = []
for (sub_id, task), (raw, epochs, info) in preprocessed_data.items():
    print(f"\nUsage 4 (alpha) benchmark: {sub_id} / {task}")
    
    for name, dss_fn in alpha_methods:
        print(f"  Testing {name}...")
        metrics = benchmark_band_extraction(epochs, ALPHA_BAND, name, dss_fn)
        metrics["subject"] = sub_id
        metrics["task"] = task
        usage4_results.append(metrics)
        
        if metrics["success"]:
            print(f"    Power: {metrics.get('band_power', '?'):.4f}, "
                  f"Runtime: {metrics['runtime_s']:.2f}s")

if usage4_results:
    u4_df = pd.DataFrame(usage4_results)
    print("\nUsage 4: Alpha Band Extraction Benchmark")
    print(u4_df[[c for c in ["denoiser", "band_power", "runtime_s", "success"]
                 if c in u4_df.columns]].to_string())
    BENCHMARK_RESULTS.extend(usage4_results)

---
## Section 9: DSS Usage 5 — Theta Band Extraction (4-7 Hz)

Same approach as alpha, targeting the theta band.

In [21]:
theta_methods = [
    ("Original (Morlet)", None),
    ("DSS + BandpassBias(4,7)", lambda n: DSS(
        bias=BandpassBias(freq_band=(4, 7), sfreq=SFREQ), n_components=n
    )),
    ("narrowband_dss(5.5Hz)", lambda n: narrowband_dss(
        sfreq=SFREQ, freq=5.5, bandwidth=3, n_components=n
    )),
    ("IterativeDSS + SpectrogramDenoiser", lambda n: IterativeDSS(
        denoiser=SpectrogramDenoiser().denoise,
        n_components=n, method="deflation", max_iter=50, random_state=42
    )),
]

usage5_results = []
for (sub_id, task), (raw, epochs, info) in preprocessed_data.items():
    print(f"\nUsage 5 (theta) benchmark: {sub_id} / {task}")
    
    for name, dss_fn in theta_methods:
        print(f"  Testing {name}...")
        metrics = benchmark_band_extraction(epochs, THETA_BAND, name, dss_fn)
        metrics["subject"] = sub_id
        metrics["task"] = task
        usage5_results.append(metrics)
        
        if metrics["success"]:
            print(f"    Power: {metrics.get('band_power', '?'):.4f}, "
                  f"Runtime: {metrics['runtime_s']:.2f}s")

if usage5_results:
    u5_df = pd.DataFrame(usage5_results)
    print("\nUsage 5: Theta Band Extraction Benchmark")
    print(u5_df[[c for c in ["denoiser", "band_power", "runtime_s", "success"]
                 if c in u5_df.columns]].to_string())
    BENCHMARK_RESULTS.extend(usage5_results)

---
## Section 10: DSS Usage 6 — Gamma ITPC Enhancement (25-40 Hz)

Enhance gamma-band phase coherence by first extracting spatially coherent
gamma components with DSS.

In [22]:
gamma_methods = [
    ("Original (Morlet ITPC)", None),
    ("DSS + BandpassBias(25,40)", lambda n: DSS(
        bias=BandpassBias(freq_band=(25, 40), sfreq=SFREQ), n_components=n
    )),
    ("narrowband_dss(33Hz)", lambda n: narrowband_dss(
        sfreq=SFREQ, freq=33, bandwidth=15, n_components=n
    )),
]

usage6_results = []
for (sub_id, task), (raw, epochs, info) in preprocessed_data.items():
    if "fast" not in task.lower():
        continue  # Gamma ITPC is FAST-specific
    
    print(f"\nUsage 6 (gamma) benchmark: {sub_id} / {task}")
    
    for name, dss_fn in gamma_methods:
        print(f"  Testing {name}...")
        metrics = {"denoiser": name, "usage": "gamma_itpc", "subject": sub_id, "task": task}
        
        t0 = time.time()
        try:
            gamma_freqs = FREQS[(FREQS >= 25) & (FREQS <= 40)]
            gamma_cycles = gamma_freqs / 2
            
            if dss_fn is None:
                # Original: ITPC on raw epochs
                itc = compute_itpc(epochs, gamma_freqs, gamma_cycles)
                occ_chs = [ch for ch in ["O1", "O2", "Oz"] if ch in epochs.ch_names]
                ch_idx = [epochs.ch_names.index(ch) for ch in occ_chs]
                t_mask = (itc.times >= 0.0) & (itc.times <= 0.2)
                gamma_itpc = itc.data[ch_idx][:, :, t_mask].mean()
            else:
                # DSS-enhanced: extract gamma components, compute ITPC on top source
                dss = dss_fn(5)
                dss.fit(epochs)
                # Both DSS and narrowband_dss return numpy sources
                sources = dss.transform(epochs)
                
                # Sources: (n_epochs, n_components, n_times) for Epochs input
                if sources.ndim == 3:
                    # Take top component: (n_epochs, 1, n_times)
                    top_source = sources[:, 0:1, :]
                    # Create single-channel EpochsArray for ITPC
                    source_info = mne.create_info(
                        ch_names=["DSS0"], sfreq=SFREQ, ch_types=["eeg"]
                    )
                    source_epochs = mne.EpochsArray(
                        top_source, source_info, tmin=epochs.tmin, verbose=False
                    )
                    itc = compute_itpc(source_epochs, gamma_freqs, gamma_cycles)
                    t_mask = (itc.times >= 0.0) & (itc.times <= 0.2)
                    gamma_itpc = itc.data[0, :, t_mask].mean()
                else:
                    gamma_itpc = np.nan
            
            metrics["gamma_itpc"] = gamma_itpc
            metrics["runtime_s"] = time.time() - t0
            metrics["success"] = True
            
        except Exception as e:
            metrics["runtime_s"] = time.time() - t0
            metrics["success"] = False
            metrics["error"] = str(e)
        
        usage6_results.append(metrics)
        if metrics["success"]:
            print(f"    ITPC: {metrics.get('gamma_itpc', '?')}, Runtime: {metrics['runtime_s']:.2f}s")

if usage6_results:
    u6_df = pd.DataFrame(usage6_results)
    print("\nUsage 6: Gamma ITPC Enhancement Benchmark")
    print(u6_df[[c for c in ["denoiser", "gamma_itpc", "runtime_s", "success"]
                 if c in u6_df.columns]].to_string())
    BENCHMARK_RESULTS.extend(usage6_results)

---
## Section 11: DSS Usage 7 — Connectivity Enhancement (wPLI)

For the AVSRT paradigm, test whether DSS pre-filtering improves
theta-band wPLI connectivity estimates.

| # | Approach | Details |
|---|----------|--------|
| 0 | Original | Laplacian + raw wPLI |
| 1 | DSS theta pre-filter | narrowband_dss(4-7 Hz) before wPLI |
| 2 | DSS bandpass pre-filter | BandpassBias(4,7) before wPLI |

In [ ]:
usage7_results = []

if not HAS_CONNECTIVITY:
    print("mne-connectivity not installed. Skipping Usage 7.")
else:
    for (sub_id, task), (raw, epochs, info) in preprocessed_data.items():
        if "avsrt" not in task.lower():
            continue
        
        print(f"\nUsage 7 (wPLI) benchmark: {sub_id} / {task}")
        
        # Original: Laplacian + wPLI
        metrics = {"denoiser": "Original (Laplacian + wPLI)", "usage": "connectivity",
                   "subject": sub_id, "task": task}
        t0 = time.time()
        try:
            con = compute_wpli(epochs)
            metrics["runtime_s"] = time.time() - t0
            metrics["success"] = con is not None
        except Exception as e:
            metrics["runtime_s"] = time.time() - t0
            metrics["success"] = False
            metrics["error"] = str(e)
        usage7_results.append(metrics)
        
        # DSS theta pre-filter + wPLI
        # Both methods use return_type='epochs' so DSS.transform() returns
        # a denoised MNE Epochs object that can be passed to compute_wpli()
        for dss_name, dss_fn in [
            ("narrowband_dss(5.5Hz) + wPLI", lambda: narrowband_dss(
                sfreq=SFREQ, freq=5.5, bandwidth=3, n_components=5, return_type="epochs"
            )),
            ("DSS+BandpassBias(4,7) + wPLI", lambda: DSS(
                bias=BandpassBias(freq_band=(4, 7), sfreq=SFREQ),
                n_components=5, return_type="epochs"
            )),
        ]:
            metrics = {"denoiser": dss_name, "usage": "connectivity",
                       "subject": sub_id, "task": task}
            t0 = time.time()
            try:
                dss = dss_fn()
                dss.fit(epochs)
                denoised = dss.transform(epochs)
                # With return_type='epochs', DSS returns an MNE Epochs object
                if hasattr(denoised, 'info'):
                    con = compute_wpli(denoised)
                    metrics["success"] = con is not None
                else:
                    metrics["success"] = False
                    metrics["error"] = "DSS did not return Epochs object"
                metrics["runtime_s"] = time.time() - t0
            except Exception as e:
                metrics["runtime_s"] = time.time() - t0
                metrics["success"] = False
                metrics["error"] = str(e)
            usage7_results.append(metrics)
            print(f"  {dss_name}: {'OK' if metrics['success'] else 'FAIL'} "
                  f"({metrics['runtime_s']:.1f}s)")
        
        break  # One subject for dev

if usage7_results:
    u7_df = pd.DataFrame(usage7_results)
    print("\nUsage 7: Connectivity Enhancement Benchmark")
    print(u7_df[[c for c in ["denoiser", "runtime_s", "success"]
                 if c in u7_df.columns]].to_string())
    BENCHMARK_RESULTS.extend(usage7_results)

---
## Section 12: Benchmark Summary

Collect all results into comparison tables and figures.

In [23]:
# Compile all benchmark results
if BENCHMARK_RESULTS:
    all_results_df = pd.DataFrame(BENCHMARK_RESULTS)
    
    print("=" * 70)
    print("COMPLETE BENCHMARK SUMMARY")
    print("=" * 70)
    
    # Summary by usage
    for usage in all_results_df["usage"].unique():
        usage_df = all_results_df[all_results_df["usage"] == usage]
        print(f"\n--- {usage.upper()} ---")
        
        # Display relevant columns
        display_cols = ["denoiser", "runtime_s", "success"]
        for col in ["snr_dB", "reduction_dB", "band_power", "gamma_itpc",
                    "n_eye_components", "max_eog_correlation"]:
            if col in usage_df.columns and usage_df[col].notna().any():
                display_cols.append(col)
        
        available_cols = [c for c in display_cols if c in usage_df.columns]
        print(usage_df[available_cols].to_string(index=False))
    
    # Save to CSV
    output_path = RESULTS_DIR / "benchmark_results.csv"
    all_results_df.to_csv(output_path, index=False)
    print(f"\nResults saved to: {output_path}")
    
else:
    print("No benchmark results collected.")

No benchmark results collected.


In [24]:
# Generate comparison figures
if BENCHMARK_RESULTS:
    all_df = pd.DataFrame(BENCHMARK_RESULTS)
    usages = all_df["usage"].unique()
    
    fig, axes = plt.subplots(1, len(usages), figsize=(5 * len(usages), 5))
    if len(usages) == 1:
        axes = [axes]
    
    for ax, usage in zip(axes, usages):
        udf = all_df[all_df["usage"] == usage]
        successful = udf[udf["success"] == True]
        
        if len(successful) > 0 and "runtime_s" in successful.columns:
            bars = ax.barh(
                range(len(successful)),
                successful["runtime_s"].values,
                color="steelblue", alpha=0.8
            )
            ax.set_yticks(range(len(successful)))
            ax.set_yticklabels(
                [d[:25] for d in successful["denoiser"].values],
                fontsize=8
            )
            ax.set_xlabel("Runtime (s)")
        
        ax.set_title(usage.replace("_", " ").title(), fontsize=10)
    
    plt.suptitle("DSS Benchmark: Runtime Comparison", fontsize=13)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "benchmark_runtime.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    print("Figures saved.")
else:
    print("No results to plot.")

No results to plot.


In [25]:
print("=" * 70)
print("NOTEBOOK COMPLETE")
print("=" * 70)
print()
print("This notebook benchmarked mne-denoise DSS methods against the original")
print("preprocessing and analysis pipelines from the SFARI EEG dataset.")
print()
print("7 Usages benchmarked:")
print("  1. Artifact removal (IterativeDSS vs ICA)")
print("  2. Line noise removal (ZapLine vs lowpass)")
print("  3. ERP enhancement (AverageBias vs simple averaging)")
print("  4. Alpha band extraction (BandpassBias/narrowband vs Morlet)")
print("  5. Theta band extraction (BandpassBias/narrowband vs Morlet)")
print("  6. Gamma ITPC enhancement (BandpassBias vs direct ITPC)")
print("  7. Connectivity enhancement (DSS pre-filter + wPLI)")
print()
if BENCHMARK_RESULTS:
    n_total = len(BENCHMARK_RESULTS)
    n_success = sum(1 for r in BENCHMARK_RESULTS if r.get("success", False))
    print(f"Total benchmarks: {n_total} ({n_success} successful)")
    print(f"Results saved to: {RESULTS_DIR}")

NOTEBOOK COMPLETE

This notebook benchmarked mne-denoise DSS methods against the original
preprocessing and analysis pipelines from the SFARI EEG dataset.

7 Usages benchmarked:
  1. Artifact removal (IterativeDSS vs ICA)
  2. Line noise removal (ZapLine vs lowpass)
  3. ERP enhancement (AverageBias vs simple averaging)
  4. Alpha band extraction (BandpassBias/narrowband vs Morlet)
  5. Theta band extraction (BandpassBias/narrowband vs Morlet)
  6. Gamma ITPC enhancement (BandpassBias vs direct ITPC)
  7. Connectivity enhancement (DSS pre-filter + wPLI)

